In [1]:
!pip install captum

In [2]:

import pandas as pd
import torch
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)

In [3]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("shivamb/real-or-fake-fake-jobposting-prediction")

print("Path to dataset files:", path)

100%|██████████| 16.1M/16.1M [00:00<00:00, 99.9MB/s]

Extracting files...


Path to dataset files: /root/.cache/kagglehub/datasets/shivamb/real-or-fake-fake-jobposting-prediction/versions/1


In [4]:
import pandas as pd
import os

# The path variable is available from the previous cell's output
file_path = os.path.join(path, 'fake_job_postings.csv')

# Load the dataset into a pandas DataFrame
df = pd.read_csv(file_path)

In [5]:
df.shape

(17880, 18)

In [6]:
df.head()

,job_id,title,location,department,salary_range,company_profile,description,requirements,benefits,telecommuting,has_company_logo,has_questions,employment_type,required_experience,required_education,industry,function,fraudulent
0,1,Marketing Intern,"US, NY, New York",Marketing,NaN,"We're Food52, and we've created a groundbreaki...","Food52, a fast-growing, James Beard Award-winn...",Experience with content management systems a m...,NaN,0,1,0,Other,Internship,NaN,NaN,Marketing,0
1,2,Customer Service - Cloud Video Production,"NZ, , Auckland",Success,NaN,"90 Seconds, the worlds Cloud Video Production ...",Organised - Focused - Vibrant - Awesome!Do you...,What we expect from you:Your key responsibilit...,What you will get from usThrough being part of...,0,1,0,Full-time,Not Applicable,NaN,Marketing and Advertising,Customer Service,0
2,3,Commissioning Machinery Assistant (CMA),"US, IA, Wever",NaN,NaN,Valor Services provides Workforce Solutions th...,"Our client, located in Houston, is actively se...",Implement pre-commissioning and commissioning ...,NaN,0,1,0,NaN,NaN,NaN,NaN,NaN,0
3,4,Account Executive - Washington DC,"US, DC, Washington",Sales,NaN,Our passion for improving quality of life thro...,THE COMPANY: ESRI – Environmental Systems Rese...,"EDUCATION: Bachelor’s or Master’s in GIS, busi...",Our culture is anything but corporate—we have ...,0,1,0,Full-time,Mid-Senior level,Bachelor's Degree,Computer Software,Sales,0
4,5,Bill Review Manager,"US, FL, Fort Worth",NaN,NaN,SpotSource Solutions LLC is a Global Human Cap...,JOB TITLE: Itemization Review ManagerLOCATION:...,QUALIFICATIONS:RN license in the State of Texa...,Full Benefits Offered,0,1,1,Full-time,Mid-Senior level,Bachelor's Degree,Hospital & Health Care,Health Care Provider,0


In [7]:
df = df[['description', 'fraudulent']].dropna()
df['description'] = df['description'].astype(str)
df['fraudulent'] = df['fraudulent'].astype(int)

# BERT Model

In [8]:
from sklearn.model_selection import train_test_split
import numpy as np

train_df, val_df = train_test_split(df, test_size=0.1, stratify=df['fraudulent'])

In [9]:
from transformers import BertTokenizerFast

tokenizer = BertTokenizerFast.from_pretrained("bert-base-uncased")

def tokenize(batch):
    return tokenizer(
        batch["description"],
        padding="max_length",
        truncation=True,
        max_length=256
    )


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

In [10]:
from datasets import Dataset

train_ds = Dataset.from_pandas(train_df)
val_ds   = Dataset.from_pandas(val_df)

train_ds = train_ds.map(tokenize, batched=True)
val_ds   = val_ds.map(tokenize, batched=True)

train_ds = train_ds.rename_column("fraudulent", "labels")
val_ds   = val_ds.rename_column("fraudulent", "labels")

train_ds.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])
val_ds.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])

Map:   0%|          | 0/16091 [00:00<?, ? examples/s]

Map:   0%|          | 0/1788 [00:00<?, ? examples/s]

In [11]:
import torch
from transformers import BertForSequenceClassification

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = BertForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=2
).to(device)


model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [12]:
import pandas as pd
import torch
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)

# Load tokenizer and model with CORRECT labels
tokenizer = AutoTokenizer.from_pretrained("microsoft/MiniLM-L12-H384-uncased")
model = AutoModelForSequenceClassification.from_pretrained(
    "microsoft/MiniLM-L12-H384-uncased",
    num_labels=2,
    id2label={0: "legitimate", 1: "fraud"},  # ✅ Class 0 = legit, Class 1 = fraud
    label2id={"legitimate": 0, "fraud": 1}   # ✅ Maps labels to IDs
)

print("✅ Model loaded with correct label mapping:")
print(f"   Class 0: {model.config.id2label[0]}")
print(f"   Class 1: {model.config.id2label[1]}")


tokenizer_config.json:   0%|          | 0.00/2.00 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/133M [00:00<?, ?B/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at microsoft/MiniLM-L12-H384-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


✅ Model loaded with correct label mapping:
   Class 0: legitimate
   Class 1: fraud


In [13]:
import torch
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")


True
NVIDIA L4


In [29]:
# ============================================================================
# PLAN B: OVERSAMPLING (RUN THIS IF CURRENT TRAINING FAILS)
# ============================================================================

from sklearn.utils import resample

print("Creating balanced training set via oversampling...")

# Separate fraud and legit
train_df_fraud = train_df[train_df['fraudulent'] == 1]
train_df_legit = train_df[train_df['fraudulent'] == 0]

print(f"Original: {len(train_df_fraud)} fraud, {len(train_df_legit)} legit")

# Oversample fraud to match legit count
train_df_fraud_oversampled = resample(
    train_df_fraud,
    n_samples=len(train_df_legit),
    random_state=42,
    replace=True
)

# Combine and shuffle
train_df_balanced = pd.concat([train_df_legit, train_df_fraud_oversampled])
train_df_balanced = train_df_balanced.sample(frac=1, random_state=42)

print(f"Balanced: {(train_df_balanced['fraudulent']==1).sum()} fraud, {(train_df_balanced['fraudulent']==0).sum()} legit")

# Recreate dataset
train_ds_balanced = Dataset.from_pandas(train_df_balanced)
train_ds_balanced = train_ds_balanced.map(tokenize, batched=True)
train_ds_balanced = train_ds_balanced.rename_column('fraudulent', 'labels')
train_ds_balanced.set_format(type='torch', columns=['input_ids', 'attention_mask', 'labels'])

# Reinitialize model
model = AutoModelForSequenceClassification.from_pretrained(
    'microsoft/MiniLM-L12-H384-uncased',
    num_labels=2,
    id2label={0: 'legitimate', 1: 'fraud'},
    label2id={'legitimate': 0, 'fraud': 1}
)

# NORMAL training (no extreme weights needed)
training_args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=3,
    per_device_train_batch_size=32,
    learning_rate=2e-5,
    weight_decay=0.01,
    logging_steps=100,
    fp16=True,
)

# Regular trainer (no custom weights)
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds_balanced,
    eval_dataset=val_ds,
)

print("\nTraining with BALANCED dataset (no extreme weights)...")
trainer.train()


Creating balanced training set via oversampling...
Original: 778 fraud, 15313 legit
Balanced: 15313 fraud, 15313 legit


Map:   0%|          | 0/30626 [00:00<?, ? examples/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at microsoft/MiniLM-L12-H384-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.



Training with BALANCED dataset (no extreme weights)...


Step,Training Loss
100,0.589000
200,0.395000
300,0.281500
400,0.222900
500,0.193900
600,0.198600
700,0.143100
800,0.143000
900,0.127000
1000,0.104500


TrainOutput(global_step=2874, training_loss=0.11058071378045557, metrics={'train_runtime': 232.4179, 'train_samples_per_second': 395.314, 'train_steps_per_second': 12.366, 'total_flos': 3026127348615168.0, 'train_loss': 0.11058071378045557, 'epoch': 3.0})

In [30]:
# ============================================================================
# EVALUATE OVERSAMPLING MODEL
# ============================================================================

print("\n" + "="*80)
print("EVALUATION - OVERSAMPLING APPROACH")
print("="*80)

model.eval()
fraud_probs = []
true_labels = []

with torch.no_grad():
    for batch in val_ds:
        inputs = {
            'input_ids': batch['input_ids'].unsqueeze(0).to(model.device),
            'attention_mask': batch['attention_mask'].unsqueeze(0).to(model.device)
        }

        outputs = model(**inputs)
        probs = torch.softmax(outputs.logits, dim=1)
        fraud_prob = probs[0][1].item()

        fraud_probs.append(fraud_prob)
        true_labels.append(batch['labels'].item())

# Find optimal threshold
from sklearn.metrics import f1_score, classification_report, confusion_matrix

best_f1 = 0
best_threshold = 0.5

for threshold in [0.3, 0.4, 0.5, 0.6, 0.7]:
    preds = [1 if p > threshold else 0 for p in fraud_probs]
    f1 = f1_score(true_labels, preds, zero_division=0)

    if f1 > best_f1:
        best_f1 = f1
        best_threshold = threshold

print(f"\nBest threshold: {best_threshold}")
predictions = [1 if p > best_threshold else 0 for p in fraud_probs]

print("\nClassification Report:")
print(classification_report(
    true_labels,
    predictions,
    target_names=['Legitimate', 'Fraud'],
    digits=3
))

cm = confusion_matrix(true_labels, predictions)
tn, fp, fn, tp = cm.ravel()

print(f"\nConfusion Matrix:")
print(f"                Predicted")
print(f"                Legit  Fraud")
print(f"Actual Legit    {tn:4d}  {fp:4d}")
print(f"       Fraud    {fn:4d}  {tp:4d}")

print(f"\n✓ Fraud Recall: {tp/(tp+fn)*100:.1f}%")
print(f"✓ Precision: {tp/(tp+fp)*100:.1f}%")
print(f"✓ F1-Score: {best_f1:.3f}")



EVALUATION - OVERSAMPLING APPROACH

Best threshold: 0.3

Classification Report:
              precision    recall  f1-score   support

  Legitimate      0.987     0.994     0.990      1701
       Fraud      0.865     0.736     0.795        87

    accuracy                          0.982      1788
   macro avg      0.926     0.865     0.893      1788
weighted avg      0.981     0.982     0.981      1788


Confusion Matrix:
                Predicted
                Legit  Fraud
Actual Legit    1691    10
       Fraud      23    64

✓ Fraud Recall: 73.6%
✓ Precision: 86.5%
✓ F1-Score: 0.795


# Testing

In [31]:
# ============================================================================
# EVALUATE ON LARGER TEST SET
# ============================================================================

# Get ALL fraud cases + equal number of legit
all_fraud = df[df['fraudulent'] == 1]
n_fraud = len(all_fraud)

# Sample equal legit for balanced test
legit_sample = df[df['fraudulent'] == 0].sample(n=n_fraud, random_state=42)

# Create test set
test_df_large = pd.concat([all_fraud, legit_sample]).sample(frac=1, random_state=42)

print(f"Large test set: {len(test_df_large)} samples")
print(f"  Fraud: {(test_df_large['fraudulent']==1).sum()}")
print(f"  Legit: {(test_df_large['fraudulent']==0).sum()}")

# Tokenize
test_ds_large = Dataset.from_pandas(test_df_large)
test_ds_large = test_ds_large.map(tokenize, batched=True)
test_ds_large = test_ds_large.rename_column('fraudulent', 'labels')
test_ds_large.set_format(type='torch', columns=['input_ids', 'attention_mask', 'labels'])

# Predict
fraud_probs_large = []
true_labels_large = []

model.eval()
with torch.no_grad():
    for batch in test_ds_large:
        inputs = {
            'input_ids': batch['input_ids'].unsqueeze(0).to(model.device),
            'attention_mask': batch['attention_mask'].unsqueeze(0).to(model.device)
        }

        outputs = model(**inputs)
        probs = torch.softmax(outputs.logits, dim=1)
        fraud_prob = probs[0][1].item()

        fraud_probs_large.append(fraud_prob)
        true_labels_large.append(batch['labels'].item())

# Evaluate with threshold 0.3
predictions_large = [1 if p > 0.3 else 0 for p in fraud_probs_large]

print("\n" + "="*80)
print(f"LARGE TEST SET EVALUATION ({len(test_df_large)} samples)")
print("="*80)

print("\nClassification Report:")
print(classification_report(
    true_labels_large,
    predictions_large,
    target_names=['Legitimate', 'Fraud'],
    digits=3
))


Large test set: 1730 samples
  Fraud: 865
  Legit: 865


Map:   0%|          | 0/1730 [00:00<?, ? examples/s]


LARGE TEST SET EVALUATION (1730 samples)

Classification Report:
              precision    recall  f1-score   support

  Legitimate      0.971     0.998     0.984       865
       Fraud      0.998     0.970     0.984       865

    accuracy                          0.984      1730
   macro avg      0.984     0.984     0.984      1730
weighted avg      0.984     0.984     0.984      1730



In [32]:
# ============================================================================
# SAVE MODEL AND RESULTS
# ============================================================================

# Save model
model.save_pretrained('./fraud_detection_final')
tokenizer.save_pretrained('./fraud_detection_final')

# Save results summary
results_summary = {
    'model': 'MiniLM-L12-H384-uncased',
    'approach': 'Oversampling (50/50 balance)',
    'test_samples': 1730,
    'test_fraud': 865,
    'test_legit': 865,
    'accuracy': 0.984,
    'fraud_precision': 0.998,
    'fraud_recall': 0.970,
    'fraud_f1': 0.984,
    'threshold': 0.3,
    'training_time': '232 seconds',
    'final_loss': 0.111
}

import json
with open('./model_results.json', 'w') as f:
    json.dump(results_summary, f, indent=2)

print("✓ Model and results saved!")


✓ Model and results saved!


# Integrated Gradients

In [35]:
from captum.attr import IntegratedGradients
import torch.nn.functional as F
import numpy as np

# ============================================================================
# INTEGRATED GRADIENTS SETUP
# ============================================================================

def forward_pass(input_ids, attention_mask):
    outputs = model(
        input_ids=input_ids,
        attention_mask=attention_mask,
    )
    # Probability of the "fraudulent = 1" class
    return F.softmax(outputs.logits, dim=1)[:, 1]

# For embedding-based attribution (more accurate)
def forward_pass_embeds(inputs_embeds, attention_mask):
    outputs = model(
        inputs_embeds=inputs_embeds,
        attention_mask=attention_mask,
    )
    return F.softmax(outputs.logits, dim=1)[:, 1]

# Initialize Integrated Gradients
ig = IntegratedGradients(forward_pass_embeds)

# ============================================================================
# HELPER FUNCTION: VISUALIZE ATTRIBUTIONS
# ============================================================================

def visualize_attributions(text, attributions, tokens, top_k=15):
    """
    Display top positive and negative attribution words
    """
    # Filter out special tokens
    filtered = [(tok, attr) for tok, attr in zip(tokens, attributions)
                if tok not in ['[CLS]', '[SEP]', '[PAD]']]

    # Sort by attribution
    sorted_attr = sorted(filtered, key=lambda x: x[1], reverse=True)

    print("\n" + "="*80)
    print("WORD ATTRIBUTION ANALYSIS")
    print("="*80)
    print(f"\nText: {text[:150]}...")

    print(f"\n🔴 Top {top_k} Fraud Indicators (positive attribution):")
    for i, (token, attr) in enumerate(sorted_attr[:top_k], 1):
        print(f"  {i:2d}. {token:20s} → {attr:+.4f}")

    print(f"\n🟢 Top {min(top_k, 5)} Legitimacy Indicators (negative attribution):")
    for i, (token, attr) in enumerate(sorted_attr[-5:], 1):
        print(f"  {i:2d}. {token:20s} → {attr:+.4f}")

# ============================================================================
# ANALYZE A FRAUD EXAMPLE
# ============================================================================

print("\n" + "="*80)
print("EXAMPLE 1: HIGH-CONFIDENCE FRAUD DETECTION")
print("="*80)

# Get a fraud example from test set
fraud_examples = test_df_large[test_df_large['fraudulent'] == 1]
fraud_example = fraud_examples.iloc[0]  # First fraud

text = fraud_example['description']
encoded = tokenizer(
    text,
    return_tensors='pt',
    truncation=True,
    max_length=256,
    padding='max_length'
).to(model.device)

# Get prediction
model.eval()
with torch.no_grad():
    fraud_prob = forward_pass(encoded['input_ids'], encoded['attention_mask']).item()

print(f"\n📊 Model Prediction: {fraud_prob:.1%} fraud probability")
print(f"🎯 Actual Label: FRAUD")
print(f"✓ Correct prediction!" if fraud_prob > 0.3 else "✗ Missed!")

# Get embeddings for IG
word_embeddings = model.base_model.embeddings.word_embeddings
inputs_embeds = word_embeddings(encoded['input_ids']).detach().requires_grad_(True)
baseline_embeds = torch.zeros_like(inputs_embeds).to(model.device)

# Calculate attributions
attributions, delta = ig.attribute(
    inputs=inputs_embeds,
    baselines=baseline_embeds,
    additional_forward_args=(encoded['attention_mask'],),
    return_convergence_delta=True,
    n_steps=50
)

# Sum across embedding dimension and normalize
attributions_sum = attributions.sum(dim=-1).squeeze(0)
attributions_norm = attributions_sum / torch.norm(attributions_sum)

# Get tokens
tokens = tokenizer.convert_ids_to_tokens(encoded['input_ids'][0])

# Visualize
visualize_attributions(text, attributions_norm.detach().cpu().numpy(), tokens, top_k=15)

print(f"\n📈 Convergence Delta: {delta.item():.6f} (closer to 0 is better)")

# ============================================================================
# ANALYZE A LEGITIMATE EXAMPLE
# ============================================================================

print("\n\n" + "="*80)
print("EXAMPLE 2: HIGH-CONFIDENCE LEGITIMATE DETECTION")
print("="*80)

# Get a legitimate example
legit_examples = test_df_large[test_df_large['fraudulent'] == 0]
legit_example = legit_examples.iloc[0]

text_legit = legit_example['description']
encoded_legit = tokenizer(
    text_legit,
    return_tensors='pt',
    truncation=True,
    max_length=256,
    padding='max_length'
).to(model.device)

# Get prediction
with torch.no_grad():
    fraud_prob_legit = forward_pass(encoded_legit['input_ids'], encoded_legit['attention_mask']).item()

print(f"\n📊 Model Prediction: {fraud_prob_legit:.1%} fraud probability")
print(f"🎯 Actual Label: LEGITIMATE")
print(f"✓ Correct prediction!" if fraud_prob_legit < 0.3 else "✗ False positive!")

# Get embeddings and attributions
inputs_embeds_legit = word_embeddings(encoded_legit['input_ids']).detach().requires_grad_(True)
baseline_embeds_legit = torch.zeros_like(inputs_embeds_legit).to(model.device)

attributions_legit, delta_legit = ig.attribute(
    inputs=inputs_embeds_legit,
    baselines=baseline_embeds_legit,
    additional_forward_args=(encoded_legit['attention_mask'],),
    return_convergence_delta=True,
    n_steps=50
)

attributions_sum_legit = attributions_legit.sum(dim=-1).squeeze(0)
attributions_norm_legit = attributions_sum_legit / torch.norm(attributions_sum_legit)
tokens_legit = tokenizer.convert_ids_to_tokens(encoded_legit['input_ids'][0])

visualize_attributions(text_legit, attributions_norm_legit.detach().cpu().numpy(), tokens_legit, top_k=15)

print(f"\n📈 Convergence Delta: {delta_legit.item():.6f}")

# ============================================================================
# ANALYZE A FALSE NEGATIVE (if any exist)
# ============================================================================

# Find missed frauds
missed_frauds_idx = []
for i, (true, pred, prob) in enumerate(zip(true_labels_large, predictions_large, fraud_probs_large)):
    if true == 1 and pred == 0:  # Missed fraud
        missed_frauds_idx.append(i)

if len(missed_frauds_idx) > 0:
    print("\n\n" + "="*80)
    print("EXAMPLE 3: FALSE NEGATIVE (MISSED FRAUD)")
    print("="*80)

    fn_idx = missed_frauds_idx[0]
    fn_example = test_df_large.iloc[fn_idx]
    text_fn = fn_example['description']

    encoded_fn = tokenizer(
        text_fn,
        return_tensors='pt',
        truncation=True,
        max_length=256,
        padding='max_length'
    ).to(model.device)

    with torch.no_grad():
        fraud_prob_fn = forward_pass(encoded_fn['input_ids'], encoded_fn['attention_mask']).item()

    print(f"\n📊 Model Prediction: {fraud_prob_fn:.1%} fraud probability (below 0.3 threshold)")
    print(f"🎯 Actual Label: FRAUD")
    print(f"✗ MISSED - Why did the model fail?")

    # Get attributions
    inputs_embeds_fn = word_embeddings(encoded_fn['input_ids']).detach().requires_grad_(True)
    baseline_embeds_fn = torch.zeros_like(inputs_embeds_fn).to(model.device)

    attributions_fn, delta_fn = ig.attribute(
        inputs=inputs_embeds_fn,
        baselines=baseline_embeds_fn,
        additional_forward_args=(encoded_fn['attention_mask'],),
        return_convergence_delta=True,
        n_steps=50
    )

    attributions_sum_fn = attributions_fn.sum(dim=-1).squeeze(0)
    attributions_norm_fn = attributions_sum_fn / torch.norm(attributions_sum_fn)
    tokens_fn = tokenizer.convert_ids_to_tokens(encoded_fn['input_ids'][0])

    visualize_attributions(text_fn, attributions_norm_fn.detach().cpu().numpy(), tokens_fn, top_k=15)

    print(f"\n📈 Convergence Delta: {delta_fn.item():.6f}")
    print("\n💡 Analysis: This fraud likely lacks obvious fraud keywords ('urgent', 'payment')")
    print("   and appears more professional/legitimate in language.")
else:
    print("\n✓ No false negatives found in test set!")

# ============================================================================
# AGGREGATE ANALYSIS: TOP FRAUD WORDS ACROSS ALL TEST SET
# ============================================================================

print("\n\n" + "="*80)
print("AGGREGATE ANALYSIS: TOP FRAUD INDICATORS ACROSS TEST SET")
print("="*80)

# Analyze 50 fraud examples
word_attributions = {}

fraud_samples = test_df_large[test_df_large['fraudulent'] == 1].head(50)

print(f"\nAnalyzing {len(fraud_samples)} fraud examples...")

for idx, row in fraud_samples.iterrows():
    text = row['description']
    encoded = tokenizer(text, return_tensors='pt', truncation=True, max_length=256, padding='max_length').to(model.device)

    inputs_embeds = word_embeddings(encoded['input_ids']).detach().requires_grad_(True)
    baseline_embeds = torch.zeros_like(inputs_embeds).to(model.device)

    attributions = ig.attribute(
        inputs=inputs_embeds,
        baselines=baseline_embeds,
        additional_forward_args=(encoded['attention_mask'],),
        n_steps=25  # Fewer steps for speed
    )

    attributions_sum = attributions.sum(dim=-1).squeeze(0)
    tokens = tokenizer.convert_ids_to_tokens(encoded['input_ids'][0])

    # Accumulate attributions by word
    for token, attr in zip(tokens, attributions_sum.detach().cpu().numpy()):
        if token not in ['[CLS]', '[SEP]', '[PAD]'] and not token.startswith('##'):
            if token not in word_attributions:
                word_attributions[token] = []
            word_attributions[token].append(attr)

# Calculate mean attribution per word
mean_attributions = {word: np.mean(attrs) for word, attrs in word_attributions.items()
                    if len(attrs) >= 3}  # At least 3 occurrences

# Sort by mean attribution
sorted_words = sorted(mean_attributions.items(), key=lambda x: x[1], reverse=True)

print("\n🔴 TOP 20 FRAUD INDICATOR WORDS (across all fraud examples):")
for i, (word, attr) in enumerate(sorted_words[:20], 1):
    count = len(word_attributions[word])
    print(f"  {i:2d}. {word:20s} → {attr:+.4f} (appeared in {count} examples)")

print("\n✓ Integrated Gradients analysis complete!")


EXAMPLE 1: HIGH-CONFIDENCE FRAUD DETECTION

📊 Model Prediction: 99.8% fraud probability
🎯 Actual Label: FRAUD
✓ Correct prediction!

WORD ATTRIBUTION ANALYSIS

Text: Apply using below link#URL_cf955625ede97e1444d64f9efbdd5c61a812c7444ce84be35525380f7549cf19#Clinical Director - Surgical ServicesPocono Health System ...

🔴 Top 15 Fraud Indicators (positive attribution):
   1. link                 → +0.7211
   2. below                → +0.3673
   3. using                → +0.2441
   4. #                    → +0.1932
   5. apply                → +0.1789
   6. health               → +0.1389
   7. we                   → +0.0964
   8. _                    → +0.0953
   9. responsible          → +0.0869
  10. be                   → +0.0784
  11. ur                   → +0.0762
  12. role                 → +0.0729
  13. cf                   → +0.0665
  14. *                    → +0.0610
  15. ##o                  → +0.0599

🟢 Top 5 Legitimacy Indicators (negative attribution):
   1. ;           

In [37]:
# Save model and tokenizer to local directory
model.save_pretrained('./model_miniLM_final')
tokenizer.save_pretrained('./model_miniLM_final')
print("✓ Model and tokenizer saved to ./model_miniLM_final")

✓ Model and tokenizer saved to ./model_miniLM_final


In [38]:
import shutil
shutil.make_archive('model_miniLM_final', 'zip', './model_miniLM_final')
print("✓ Model zipped for GitHub upload: model_miniLM_final.zip")


✓ Model zipped for GitHub upload: model_miniLM_final.zip
